# Step 11: Customer Churn API Demonstration

This notebook demonstrates how to query the deployed FastAPI application.

Before running it, start the application locally:

```bash
uvicorn app.main:app --host 0.0.0.0 --port 8000
```

The same notebook can query a cloud deployment by changing `BASE_URL`.


In [1]:
import requests
import pandas as pd

BASE_URL = "http://localhost:8000"


## Confirm application health

In [2]:
health_response = requests.get(f"{BASE_URL}/health", timeout=10)
health_response.raise_for_status()
health_response.json()


{'status': 'ok',
 'model_loaded': True,
 'model_version': 'Baseline Random Forest'}

## Submit a single prediction

In [3]:
customer = {
    "credit_score": 650,
    "age": 42,
    "tenure": 5,
    "balance": 125000.0,
    "estimated_salary": 85000.0,
    "geography": "Germany",
    "gender": "Female"
}

response = requests.post(
    f"{BASE_URL}/predict",
    json=customer,
    timeout=10
)
response.raise_for_status()
prediction = response.json()
prediction


{'prediction': 1,
 'prediction_label': 'Exited',
 'churn_probability': 0.645856,
 'model_version': 'Baseline Random Forest',
 'request_id': 'a2f72920-c304-4faa-9426-995851fde3d5'}

The response includes the predicted class, churn probability, model version, and a request ID that can be used to trace the request in application logs.

## Submit a batch prediction

In [4]:
batch = {
    "customers": [
        customer,
        {
            "credit_score": 790,
            "age": 29,
            "tenure": 2,
            "balance": 25000.0,
            "estimated_salary": 110000.0,
            "geography": "France",
            "gender": "Male"
        }
    ]
}

batch_response = requests.post(
    f"{BASE_URL}/predict/batch",
    json=batch,
    timeout=10
)
batch_response.raise_for_status()

pd.DataFrame(batch_response.json()["predictions"])


,prediction,prediction_label,churn_probability,model_version,request_id
0,1,Exited,0.645856,Baseline Random Forest,c79af741-654d-4b3e-b356-428360a62b79
1,0,Stayed,0.120211,Baseline Random Forest,c79af741-654d-4b3e-b356-428360a62b79


## Demonstrate a controlled validation error

In [5]:
invalid_customer = {**customer, "age": 12}

invalid_response = requests.post(
    f"{BASE_URL}/predict",
    json=invalid_customer,
    timeout=10
)

print("Status:", invalid_response.status_code)
invalid_response.json()


Status: 422


{'error': 'validation_error',
 'details': [{'type': 'greater_than_equal',
   'loc': ['body', 'age'],
   'msg': 'Input should be greater than or equal to 18',
   'input': 12,
   'ctx': {'ge': 18}}],
 'request_id': '70def28f-9805-475b-a45a-1587a2594d59'}

## Why the API is useful

The API separates model inference from the notebook used to train the model. A website, CRM, internal tool, or scheduled batch process can request predictions without loading scikit-learn directly.

The browser UI at the root endpoint provides a simple way to test individual customers, while `/docs` supports technical testing through Swagger. The batch endpoint supports operational scoring workflows, and the request ID makes debugging easier when the service is deployed.
